# Compare sparse trial-state circuits for stretched N₂



This notebook complements the trial-state chapter by rendering the logical circuits generated for the one-, two-, and four-determinant trial states.



Keep `tutorial_prepare_trial_state.py` and `tutorial_choose_active_space.py` in the same folder as this notebook. The tested Python scripts define the scientific workflow; this notebook calls those functions rather than duplicating the chemistry or circuit synthesis.

## Configure the notebook environment

Use the QDK circuit widget for deterministic inline rendering and IPython display helpers for labels and comparison tables.

The setup cell silences QDK/Chemistry library logs so the circuit comparisons remain easy to read. Change `Logger.LogLevel.off` to `Logger.LogLevel.info` and rerun the cell to see detailed calculation logs.

In [ ]:
from IPython.display import Markdown, display
from qdk.widgets import Circuit
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

## Import circuit definitions



Import the reusable trial-state and circuit-inspection functions from the adjacent Python script. This keeps notebook execution synchronized with the downloadable command-line example and its tests.

In [ ]:
from tutorial_prepare_trial_state import (
    circuit_statistics,
    leading_determinant_contributions,
    print_trial_state_results,
    run_trial_state_workflow,
)

## Build the trial states



Run the tested workflow once to construct the one-, two-, and four-determinant PMC trial states and their QDK/Chemistry sparse-isometry logical circuits.



Fidelity measures trial-state quality, while preparation logical gate counts measure preparation cost. Keep these quantities separate when comparing the circuits.

In [ ]:
determinant_counts = (1, 2, 4)
result = run_trial_state_workflow(determinant_counts=determinant_counts)

print_trial_state_results(result)

## Render individual circuits



Render each generated logical circuit with the same widget and labeling. Compare the visible superposition and entangling structure as determinants are added.

In [ ]:
for trial_state in result.trial_states:
    display(Markdown(f"### {trial_state.num_determinants}-determinant trial state"))
    display(Circuit(trial_state.circuit.get_qsharp_circuit()))

## Why some wires have no preparation gates



The compute register begins with every occupation qubit in $\lvert 0\rangle$. A wire may need no state-preparation gate when the selected sparse trial wavefunction does not require its occupation bit to change or become entangled. Leaving that wire untouched is itself the correct preparation operation: the qubit remains in the required zero state.



An idle wire in the state-preparation circuit is not an unused qubit in the full algorithm. Every compute qubit represents an active spin orbital and belongs to the register on which the active-space Hamiltonian acts. During QPE, controlled time evolution under that Hamiltonian can couple the initially prepared determinant support to other configurations in the physical fixed-electron-number sector. Those configurations may involve occupation qubits that required no gates during preparation.



The full compute register is therefore determined by the encoded active-space problem, whereas the visible preparation gates are determined only by the chosen trial wavefunction and the synthesis method. Removing an idle preparation wire would change the Hamiltonian representation rather than merely simplify state preparation.

## Compare circuit variants



The table compares determinant support, fidelity, compute-qubit count, preparation logical gate count, and logical gate-family counts without relying on visual inspection alone.



In these decomposed circuits, X gates flip occupation bits, H gates create basis-state superpositions, S and Rz gates set relative phases, and CNOT gates correlate occupations across wires. The exact decomposition depends on the state-preparation implementation.

In [ ]:
comparison_rows = [
    (
        trial_state.num_determinants,
        trial_state.fidelity,
        trial_state.num_compute_qubits,
        trial_state.num_logical_gates,
        trial_state.logical_gate_counts,
    )
    for trial_state in result.trial_states
]

table_rows = [
    "| Determinants | Fidelity | Compute qubits | Preparation logical gate count | Logical gate-family counts |",
    "|---:|---:|---:|---:|:---|",
]

for determinants, fidelity, qubits, gates, gate_counts in comparison_rows:
    table_rows.append(
        f"| {determinants} | {fidelity:.6f} | {qubits} | {gates} | `{gate_counts}` |"
    )

display(Markdown("\n".join(table_rows)))

## Validate circuit construction



These assertions check the requested determinant supports, common register dimensions, nonempty operations, and agreement between stored and independently recomputed circuit statistics.

In [ ]:
assert [state.num_determinants for state in result.trial_states] == list(determinant_counts)
assert len({state.num_compute_qubits for state in result.trial_states}) == 1
assert all(state.num_compute_qubits > 0 for state in result.trial_states)
assert all(state.num_logical_gates > 0 for state in result.trial_states)

for trial_state in result.trial_states:
    assert circuit_statistics(trial_state.circuit) == (
        trial_state.num_compute_qubits,
        trial_state.num_logical_gates,
        trial_state.logical_gate_counts,
    )
    assert sum(trial_state.logical_gate_counts.values()) == trial_state.num_logical_gates

## Test notebook-compatible functions



Exercise the same reusable helpers covered by the adjacent Python tests. These checks verify fidelity bounds and the ordering and normalization information reported for leading determinants.

In [ ]:
leading = leading_determinant_contributions(
    result.active_space_result.refined_casci_wavefunction, max_determinants=4
)
assert len(leading) == 4

assert all(
    round(leading[index].weight, 12)
    >= round(leading[index + 1].weight, 12)
    for index in range(len(leading) - 1)
)

assert all(
    leading[index].cumulative_weight <= leading[index + 1].cumulative_weight
    for index in range(len(leading) - 1)
)

assert all(0.0 <= state.fidelity <= 1.0 for state in result.trial_states)

## Interpretation

Record the fidelity and preparation logical gate count for each trial state in your lab notebook. Explain which added gate families account for the increasing circuit cost. Compare the marginal fidelity gain per additional logical gate from one to two determinants and from two to four determinants, then name one additional criterion that could affect the choice and explain whether it would change your preferred trial state.